# 02 · Preparación de la tabla del modelo

La pregunta del TFM es cómo ha cambiado entre 2012 y 2022 la relación del compromiso del alumno y de sus recursos de estudio con su nota de matemáticas, según el perfil de su centro. Las variables ya están construidas en los EDA del objetivo y unidas por alumno en `tabla_por_alumno` (`src/tabla_alumno.py`); `01_EDA_variables_modelo` describe sus huecos. Este notebook fija las decisiones que faltan antes de estimar y guarda la tabla de la que parten los modelos:

* ¿Qué perfiles de centro tienen centros suficientes en cada edición? (sección 1)
* ¿Cuántos alumnos tienen las dos variables de interés y entran en el modelo? (sección 2)
* ¿Cuántos huecos quedan en los controles, que llevarán un indicador de «sin dato»? (sección 3)
* ¿Con qué escala entran las variables en la regresión? (sección 4)
* ¿Cómo quedan las matrices de la regresión y de XGBoost? (sección 5)

Las funciones están en `src/preparacion_modelo.py`. Se guardan en `data/processed/` tres ficheros (sección 6):

* **`tabla_modelo_alumno.parquet`**: la tabla por alumno en sus unidades originales, sin imputar y sin codificar, con cuatro marcas que dicen quién entra en el modelo y qué perfiles son poco fiables.
* **`parametros_escala.parquet`**: media y desviación típica de cada variable continua.
* **`diccionario_tabla_modelo.csv`**: qué es cada columna y en qué unidades.

Las matrices no se guardan: `matriz_regresion` y `matriz_xgboost` las reconstruyen desde esa tabla en cada notebook de modelado, de modo que la regresión, XGBoost y el agente parten de los mismos alumnos y los mismos valores.

## 0. Configuración

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.append(os.path.abspath('../../'))   # para importar `src`
from src.preparacion_modelo import (COMPROBACIONES, CONTROLES, DE_INTERES, MARCAS, MINIMO_CENTROS, SIN_TIPIFICAR, centros_por_perfil,
                                    diccionario_columnas, marcar_universo, matriz_regresion, matriz_xgboost, parametros_escala)
from src.tabla_alumno import tabla_por_alumno

pd.set_option('display.width', 200)

EDICIONES = [2012, 2015, 2018, 2022]
RUTA_RAW_BASE = '../../data/raw'
RUTA_INTERMEDIOS = '../../data/intermediate'
RUTA_PROCESADOS = '../../data/processed'


def media_ponderada(valores, pesos):
    """Media de `valores` ponderada con `pesos`, sin contar los vacíos."""
    con_dato = valores.notna()
    return np.average(valores[con_dato], weights=pesos[con_dato]) if con_dato.any() else np.nan


# Una fila por alumno con perfil de centro de los 12 países del panel común, con las cuatro marcas del universo
tabla = marcar_universo(tabla_por_alumno(RUTA_RAW_BASE, RUTA_INTERMEDIOS))
print(f"{len(tabla)} alumnos de {tabla['CNT'].nunique()} países y {tabla['centro'].nunique()} centros-edición")

408999 alumnos de 12 países y 16591 centros-edición


## 1. Perfiles de centro

Un perfil-edición con menos de 30 centros (`MINIMO_CENTROS`, el umbral de `01_EDA_variables_modelo`) da resultados poco representativos: unos pocos centros pesan mucho en su pendiente. La regla, fijada antes de estimar, es:

* si un perfil no llega a 30 centros **en ninguna edición**, queda fuera del modelo (`perfil_excluido`);
* si no llega **en alguna edición**, entra, pero sus resultados de esa edición se marcan como poco fiables (`perfil_poco_representativo`) y no se interpretan.

In [2]:
centros = centros_por_perfil(tabla)
estado = pd.Series('entra', index=centros.index)
estado[(centros < MINIMO_CENTROS).any(axis=1)] = 'poco representativo en alguna edición'
estado[(centros < MINIMO_CENTROS).all(axis=1)] = 'excluido'
display(centros.assign(estado=estado).sort_values(['estado', 'perfil']))
print(estado.value_counts().to_string())
alumnos_excluidos = tabla.groupby('year')['perfil_excluido'].agg(alumnos='sum', porcentaje=lambda serie: 100 * serie.mean())
print('\nAlumnos de perfiles excluidos, por edición:')
print(alumnos_excluidos.round(2).T)

year,2012,2015,2018,2022,estado
perfil,,,,,
CON_RUR_PEQ,89,49,87,101,entra
CON_URB_MACRO,264,142,294,296,entra
CON_URB_MED,189,98,172,188,entra
CON_URB_PEQ,120,92,148,161,entra
PRI_URB_MACRO,107,89,107,115,entra
PUB_RUR_MACRO,153,125,120,126,entra
PUB_RUR_MED,393,221,374,316,entra
PUB_RUR_PEQ,725,561,734,623,entra
PUB_URB_MACRO,878,665,775,747,entra


entra                                    11
poco representativo en alguna edición     4
excluido                                  3

Alumnos de perfiles excluidos, por edición:
year          2012    2015    2018    2022
alumnos     502.00  368.00  608.00  466.00
porcentaje    0.43    0.47    0.55    0.44


**Qué se ve.**

* **11 de los 18 perfiles tienen 30 centros o más en las cuatro ediciones**: todos los públicos, los concertados urbanos, el concertado rural pequeño y el privado urbano grande.
* **Quedan fuera los tres privados rurales** (`PRI_RUR_MACRO`, `PRI_RUR_MED` y `PRI_RUR_PEQ`), que no pasan de 3, 6 y 22 centros en ninguna edición. Son pocos alumnos: entre el 0,43 % y el 0,55 % de cada edición.
* **Cuatro perfiles entran con alguna edición poco fiable**:
  * `CON_RUR_MACRO`: 13 centros en 2015 y 28 en 2018;
  * `CON_RUR_MED`: 28 en 2012 y 22 en 2015;
  * `PRI_URB_MED`: 29 en 2015;
  * `PRI_URB_PEQ`: 25 en 2015.

  Sus resultados de esas ediciones se dan, pero marcados como poco fiables, y no sirven para hablar de un cambio entre ediciones. 2015 es la edición con menos centros en casi todos los perfiles.

## 2. Alumnos con las variables de interés

Las dos variables de interés, el índice de compromiso y los recursos de estudio, no se imputan: entran los alumnos que tienen las dos (`con_variables_interes`) y cuyo perfil no está excluido (`en_modelo`). `01_EDA_variables_modelo` (secciones 4 y 5) muestra que los alumnos sin dato sacan menos nota, así que las relaciones del modelo valen para los alumnos que contestaron. Se cuentan por edición los que faltan por cada variable y los que entran, y por perfil el % que entra.

In [3]:
resumen = tabla.groupby('year').agg(
    alumnos=('CNTSTUID', 'size'),
    sin_indice_compromiso=('Indice_Compromiso_Competencia', lambda serie: serie.isna().sum()),
    sin_recursos_estudio=('recursos_estudio', lambda serie: serie.isna().sum()),
    sin_alguna=('con_variables_interes', lambda serie: (~serie).sum()),
    en_modelo=('en_modelo', 'sum'))
resumen['% en modelo'] = 100 * resumen['en_modelo'] / resumen['alumnos']
display(resumen.round(2))
print(f"En el modelo: {tabla['en_modelo'].sum()} alumnos de {len(tabla)} ({100 * tabla['en_modelo'].mean():.2f} %)")

porcentaje_perfil = (100 * tabla[~tabla['perfil_excluido']].groupby(['perfil', 'year'])['en_modelo'].mean()).unstack('year')
print('\n% de alumnos que entran en el modelo, por perfil y edición (mínimo y máximo de cada edición):')
print(porcentaje_perfil.agg(['min', 'idxmin', 'max', 'idxmax']))

,alumnos,sin_indice_compromiso,sin_recursos_estudio,sin_alguna,en_modelo,% en modelo
year,,,,,,
2012,115762,41096,1803,41588,73843,63.79
2015,78136,8522,1829,8979,68795,88.05
2018,110023,39572,3158,39786,69878,63.51
2022,105078,9920,13945,18436,86266,82.10


En el modelo: 298782 alumnos de 408999 (73.05 %)



% de alumnos que entran en el modelo, por perfil y edición (mínimo y máximo de cada edición):
year             2012         2015           2018           2022
min         62.892955    82.730455      39.890181      75.727299
idxmin    PUB_RUR_MED  CON_RUR_PEQ  PUB_RUR_MACRO  PUB_RUR_MACRO
max         65.247312    95.376712      76.635514      88.863287
idxmax  CON_URB_MACRO  PRI_URB_PEQ    PRI_URB_PEQ    PRI_URB_MED


**Qué se ve.**

* **Entran en el modelo 298.782 alumnos**, el 73,05 % del universo: el 63,79 % en 2012, el 88,05 % en 2015, el 63,51 % en 2018 y el 82,10 % en 2022.
* **En 2012 y 2018 casi todos los que faltan son por el índice de compromiso** (41.096 y 39.572 alumnos sin índice), por las horas de deberes: en 2012 la pregunta estaba en una forma rotada del cuestionario y en 2018 en un cuestionario opcional.
* **En 2022 pesan más los recursos de estudio** (13.945 alumnos sin ellos, frente a 9.920 sin índice), por las preguntas de tipos de libros (`01_EDA_recursos_alumno`).
* **El % que entra apenas cambia con el perfil en 2012** (del 62,89 % al 65,25 %), pero sí en las demás ediciones. En 2018 va del 39,89 % de `PUB_RUR_MACRO` al 76,64 % de `PRI_URB_PEQ`. Los alumnos que faltan sacan menos nota en todos los perfiles (`01_EDA_variables_modelo`, sección 5), y en 2022 la diferencia es mayor en los públicos: las pendientes de cada perfil valen para los alumnos que contestaron.

## 3. Huecos de los controles

En los controles y las comprobaciones, un hueco no deja fuera al alumno: se rellena con la media y se añade un indicador `sin_<variable>` (1 = sin dato), de modo que la pendiente de la variable sale solo de los alumnos que la tienen y el indicador recoge en qué se diferencian los demás. `01_EDA_variables_modelo` (sección 4) muestra que en la dotación del centro los alumnos sin dato se parecen a los demás, y que en las variables del alumno no. Se mide el % sin dato de cada variable entre los alumnos del modelo.

In [4]:
universo = tabla[tabla['en_modelo']]
huecos = (100 * universo[CONTROLES + COMPROBACIONES].isna().groupby(universo['year']).mean()).T
huecos['todas'] = 100 * universo[CONTROLES + COMPROBACIONES].isna().mean()
display(huecos.round(2))
print('Variables sin ningún hueco entre los alumnos del modelo:', list(huecos.index[huecos['todas'] == 0]))

year,2012,2015,2018,2022,todas
HISEI_TREND,1.86,4.40,3.75,5.44,3.92
PAREDINT_TREND,2.54,1.60,1.11,1.11,1.58
riqueza_hogar,0.75,2.82,3.89,0.54,1.90
libros,0.79,0.34,0.26,0.00,0.33
sexo_femenino,0.00,0.00,0.00,0.02,0.01
inmigrante,1.74,1.30,1.20,1.64,1.48
composicion_social_centro,0.00,0.00,0.00,0.00,0.00
dotacion_centro,8.49,4.45,4.05,9.14,6.71
deberes_en_u,0.00,0.00,0.00,0.00,0.00
SKIPPING,0.00,0.00,0.00,0.00,0.00


Variables sin ningún hueco entre los alumnos del modelo: ['composicion_social_centro', 'deberes_en_u', 'SKIPPING', 'TARDYSD', 'escs_centro']


**Qué se ve.**

* **Los huecos de los controles son pequeños.** Entre los alumnos del modelo, el origen social, el sexo y la inmigración faltan como mucho al 5,44 % (la ocupación de los padres en 2022). La dotación del centro falta al 6,71 % (el 9,14 % en 2022).
* **La escasez del centro falta al 16,40 % en 2022**, porque Corea y Eslovaquia no la tienen (`01_EDA_variables_modelo`). Como solo entra en una comprobación, su indicador recoge esos dos países.
* **Cinco variables no tienen ningún hueco** y no llevan indicador:
  * la composición social y el `ESCS` medio del centro, que se calculan por centro con los alumnos que tienen dato;
  * los tres componentes del índice, que todo alumno con índice tiene.

  Quedan ocho indicadores de «sin dato».

## 4. Escala de la regresión

Las variables continuas se tipifican para que cada coeficiente se lea como «puntos de nota por una desviación típica», igual que en los EDA. La media y la desviación típica se calculan con `SENWT` y con los alumnos del modelo de las **cuatro ediciones juntas**: con una escala común, un cambio del coeficiente entre ediciones es un cambio de la relación con la nota y no del reparto de la variable en esa edición. Hay dos excepciones:

* las variables de 0 o 1 y la impuntualidad (`SIN_TIPIFICAR`) quedan en su escala, que ya se lee directamente;
* la escasez del centro se tipifica dentro de cada edición, porque cada edición la mide con su propia escala (`01_EDA_recursos_centro`).

El índice de compromiso ya tiene media 0 y desviación típica 1 en el panel; se tipifica de nuevo con los alumnos del modelo, que son casi los mismos. Los parámetros se guardan para pasar un coeficiente a las unidades originales: una desviación típica de recursos de estudio son tantos recursos como diga su `desviacion_tipica`.

In [5]:
parametros = parametros_escala(tabla)
display(parametros.round(3))

matriz = matriz_regresion(tabla, parametros)
tipificadas = [variable for variable in DE_INTERES + CONTROLES + COMPROBACIONES if variable not in SIN_TIPIFICAR]
sin_hueco = {variable: matriz[f'sin_{variable}'] == 0 if f'sin_{variable}' in matriz else pd.Series(True, index=matriz.index) for variable in tipificadas}
comprobacion = pd.DataFrame({variable: {'media': media_ponderada(matriz[variable].where(sin_hueco[variable]), matriz['SENWT']),
                                        'desviación típica': np.sqrt(media_ponderada(matriz[variable].where(sin_hueco[variable]) ** 2, matriz['SENWT']))}
                             for variable in tipificadas}).T
print('Con dato, ya tipificadas (todas las ediciones; la escasez, con media 0 en cada edición):')
print(comprobacion.round(3))

medias_edicion = pd.DataFrame({variable: matriz.groupby('year').apply(lambda grupo: media_ponderada(grupo[variable], grupo['SENWT']), include_groups=False)
                               for variable in DE_INTERES}).T
print('\nMedia de las variables de interés tipificadas, por edición:')
print(medias_edicion.round(3))

,variable,edicion,media,desviacion_tipica
0,Indice_Compromiso_Competencia,todas,0.011,0.997
1,recursos_estudio,todas,5.295,0.939
2,HISEI_TREND,todas,52.739,21.856
3,PAREDINT_TREND,todas,13.909,2.340
4,riqueza_hogar,todas,3.269,1.234
5,libros,todas,3.383,1.417
6,composicion_social_centro,todas,0.034,0.459
7,dotacion_centro,todas,-0.184,0.911
8,escs_centro,todas,-0.042,0.547
9,escasez_centro,2012,-0.098,0.860


Con dato, ya tipificadas (todas las ediciones; la escasez, con media 0 en cada edición):
                               media  desviación típica
Indice_Compromiso_Competencia    0.0                1.0
recursos_estudio                 0.0                1.0
HISEI_TREND                     -0.0                1.0
PAREDINT_TREND                  -0.0                1.0
riqueza_hogar                    0.0                1.0
libros                          -0.0                1.0
composicion_social_centro       -0.0                1.0
dotacion_centro                 -0.0                1.0
escs_centro                      0.0                1.0
escasez_centro                   0.0                1.0

Media de las variables de interés tipificadas, por edición:
year                            2012   2015   2018   2022
Indice_Compromiso_Competencia  0.196  0.005 -0.066 -0.105
recursos_estudio              -0.006  0.008 -0.054  0.043


**Qué se ve.**

* **Una desviación típica de recursos de estudio son 0,939 recursos**, con media 5,295 de 6. Este es el factor para pasar un coeficiente a «puntos por cada recurso más».
* **Las demás desviaciones típicas**:
  * ocupación de los padres: 21,856 puntos ISEI;
  * estudios de los padres: 2,340 años;
  * riqueza del hogar: 1,234 bienes;
  * libros en casa: 1,417 tramos.
* **El índice de compromiso apenas cambia al tipificarlo de nuevo**: con los alumnos del modelo tiene media 0,011 y desviación típica 0,997.
* **Tras tipificar, todas las variables tienen media 0 y desviación típica 1 con dato.** En la escasez, eso vale en cada edición.
* **La escala común conserva la evolución entre ediciones.** El índice de compromiso pasa de 0,196 en 2012 a −0,105 en 2022 (desde 2015, sobre todo porque el tercil central de deberes se hace más pequeño: `01_EDA_indice_de_compromiso_competencia`, sección 5.7.4). Los recursos de estudio se quedan entre −0,054 y 0,043.

## 5. Matrices de los modelos

* **Regresión** (`matriz_regresion`): alumnos del modelo, variables tipificadas, controles sin huecos y sus indicadores. El perfil y el país quedan como texto: el efecto fijo de país y las interacciones de las variables de interés con el perfil se construyen en el notebook de la regresión, que estima una regresión por edición y valor plausible y las combina con la regla de Rubin.
* **XGBoost** (`matriz_xgboost`): los mismos alumnos y variables en sus unidades originales y con sus huecos, que XGBoost reparte solo en cada división; el perfil y el país pasan a categoría (`enable_categorical=True`), sin crear columnas.

Se comprueba que las dos tienen los mismos alumnos, que la de la regresión no tiene huecos fuera de los valores plausibles que no existen (PV6 a PV10 en 2012) y que la de XGBoost conserva los valores originales.

In [6]:
xgboost = matriz_xgboost(tabla)
valores_plausibles = [columna for columna in matriz.columns if columna.startswith('PV')]
indicadores = [columna for columna in matriz.columns if columna.startswith('sin_')]
print(f'Regresión: {matriz.shape[0]} alumnos y {matriz.shape[1]} columnas; indicadores de sin dato: {indicadores}')
print(f'XGBoost:   {xgboost.shape[0]} alumnos y {xgboost.shape[1]} columnas')
print('Mismos alumnos y en el mismo orden:', matriz[['CNT', 'CNTSTUID', 'year']].equals(xgboost[['CNT', 'CNTSTUID', 'year']]))
print('Huecos en la regresión fuera de los valores plausibles:', int(matriz.drop(columns=valores_plausibles).isna().sum().sum()))
print('Valores plausibles vacíos fuera de PV6-PV10 de 2012:',
      int(matriz.loc[matriz['year'] != 2012, valores_plausibles].isna().sum().sum() + matriz.loc[matriz['year'] == 2012, [f'PV{numero}MATH' for numero in range(1, 6)]].isna().sum().sum()))
originales = tabla[tabla['en_modelo']].reset_index(drop=True)
print('XGBoost con los valores originales:', all(xgboost[variable].equals(originales[variable]) for variable in DE_INTERES + CONTROLES + COMPROBACIONES))
print('\nAlumnos del modelo por edición:', matriz.groupby('year').size().to_dict())

Regresión: 298782 alumnos y 62 columnas; indicadores de sin dato: ['sin_HISEI_TREND', 'sin_PAREDINT_TREND', 'sin_riqueza_hogar', 'sin_libros', 'sin_sexo_femenino', 'sin_inmigrante', 'sin_dotacion_centro', 'sin_escasez_centro']
XGBoost:   298782 alumnos y 56 columnas
Mismos alumnos y en el mismo orden: True
Huecos en la regresión fuera de los valores plausibles: 0
Valores plausibles vacíos fuera de PV6-PV10 de 2012: 0


XGBoost con los valores originales: True

Alumnos del modelo por edición: {2012: 73843, 2015: 68795, 2018: 69878, 2022: 86266}


**Qué se ve.**

* **Las dos matrices tienen los mismos 298.782 alumnos, en el mismo orden**: 73.843 en 2012, 68.795 en 2015, 69.878 en 2018 y 86.266 en 2022.
* **La de la regresión no tiene huecos** fuera de los valores plausibles PV6 a PV10 de 2012, que no existen. Tiene 62 columnas con los ocho indicadores.
* **La de XGBoost conserva los valores originales** de todas las variables y tiene 56 columnas.

## 6. Guardado

La tabla se guarda **sin imputar y sin codificar**, con todas las filas (también las que no entran en el modelo) y las cuatro marcas. Es la que leen los notebooks de modelado y el agente:

* para el agente, cada columna conserva sus unidades originales (recursos de 0 a 6, horas de deberes, años de estudios de los padres...), y `diccionario_tabla_modelo.csv` dice qué es cada una;
* una versión imputada no añadiría información: el valor imputado es la media de todos, y lo único que dice es que el dato falta, que ya se ve en el hueco. Además, el agente podría tomarlo por un dato real;
* la matriz de la regresión se reconstruye con `matriz_regresion(tabla, parametros)` en un instante, así que guardarla duplicaría la tabla con el riesgo de que las dos versiones se separen.

In [7]:
os.makedirs(RUTA_PROCESADOS, exist_ok=True)
rutas = {'tabla': os.path.join(RUTA_PROCESADOS, 'tabla_modelo_alumno.parquet'),
         'parametros': os.path.join(RUTA_PROCESADOS, 'parametros_escala.parquet'),
         'diccionario': os.path.join(RUTA_PROCESADOS, 'diccionario_tabla_modelo.csv')}
diccionario = diccionario_columnas(tabla)
tabla.to_parquet(rutas['tabla'], index=False)
parametros.to_parquet(rutas['parametros'], index=False)
diccionario.to_csv(rutas['diccionario'], index=False, encoding='utf-8')

releida = pd.read_parquet(rutas['tabla'])
print('Tabla releída igual:', releida.equals(tabla.reset_index(drop=True)))
print('Misma matriz desde la tabla releída:', matriz_regresion(releida, pd.read_parquet(rutas['parametros'])).equals(matriz))
for nombre, ruta in rutas.items():
    print(f'{nombre}: {ruta} ({os.path.getsize(ruta) / 1e6:.1f} MB)')
print(f'{len(tabla)} alumnos y {tabla.shape[1]} columnas; {len(diccionario)} columnas descritas, por papel:')
print(diccionario['papel'].value_counts().to_string())

Tabla releída igual: True


Misma matriz desde la tabla releída: True
tabla: ../../data/processed\tabla_modelo_alumno.parquet (62.0 MB)
parametros: ../../data/processed\parametros_escala.parquet (0.0 MB)
diccionario: ../../data/processed\diccionario_tabla_modelo.csv (0.0 MB)
408999 alumnos y 60 columnas; 60 columnas descritas, por papel:
papel
otras notas               20
objetivo                  10
identificador              6
comprobación               5
marca                      4
control: origen social     4
peso                       2
control: alumno            2
descriptiva                2
interés                    2
control: centro            2
grupo                      1


**Qué se ve.**

* **La tabla tiene 408.999 alumnos y 60 columnas, todas descritas en el diccionario**:
  * 30 de notas: 10 valores plausibles de matemáticas, el objetivo, y 20 de lectura y ciencias;
  * 2 variables de interés, 8 controles y 5 comprobaciones;
  * 2 descriptivas, 4 marcas, 2 pesos, 6 identificadores y el perfil.
* **Al releerla da la misma tabla, y desde ella la misma matriz de la regresión.**
* **Ocupa 62,0 MB**, por los 30 valores plausibles.

## Hallazgos

* **El universo del modelo**: 298.782 alumnos de los 12 países del panel común (el 73,05 % de los que tienen perfil de centro). Las variables de interés no se imputan, así que faltan sobre todo por las horas de deberes en 2012 y 2018 y por los recursos de estudio en 2022.
* **Perfiles**: entran 15 de los 18.
  * Los tres privados rurales quedan fuera porque no llegan a 30 centros en ninguna edición; son menos del 0,6 % de los alumnos.
  * `CON_RUR_MACRO`, `CON_RUR_MED`, `PRI_URB_MED` y `PRI_URB_PEQ` tienen alguna edición con menos de 30 centros, casi siempre 2015. Esos resultados se marcan como poco fiables.
* **Controles**: ocho indicadores de «sin dato».
  * Los huecos son pequeños, salvo la dotación del centro en 2012 y 2022 (8,49 % y 9,14 %).
  * También la escasez en 2022 (16,40 %, por Corea y Eslovaquia), que solo entra en una comprobación.
* **Escala**: las variables continuas se tipifican con `SENWT` y las cuatro ediciones juntas, así que un coeficiente es «puntos por desviación típica» con la misma escala en todas las ediciones. `parametros_escala.parquet` guarda el factor para volver a las unidades originales: por ejemplo, 0,939 recursos de estudio por desviación típica.
* **Qué se guarda**: una sola tabla, sin imputar y sin codificar, con las marcas de quién entra en el modelo y un diccionario de columnas.
  * La regresión y XGBoost la leen y construyen su matriz con `matriz_regresion` y `matriz_xgboost`.
  * El agente la lee tal cual, con las variables en sus unidades.

**Siguiente paso**: la regresión por edición, con efecto fijo de país, interacciones de las dos variables de interés con el perfil, errores agrupados por centro y valores plausibles combinados con Rubin. Después, XGBoost con SHAP sobre la misma muestra.